In [1]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VBM_MIN, CBM_MAX = 1.23, 0.0
TOL = 0
REQUIRE_VBM = True


HERE = Path.cwd()
candidates = [HERE / "data",
              HERE,
              HERE / "nootebooks" / "data" / "data",
              HERE / "notebook" / "data"
              ]

DATA_DIR = next((p for p in candidates if (p / "EDGE_Cal.csv").exists()), None)
assert DATA_DIR is not None, "EDGE_Cal.csv not found. Run the extraction  notebook first."

IN_PATH = DATA_DIR / "EDGE_Cal.csv"
OUT_PATH = DATA_DIR / "BandEdge.csv"
FIG_DIR = DATA_DIR.parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", IN_PATH)
print("Output:", OUT_PATH)

Input: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\EDGE_Cal.csv
Output: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\BandEdge.csv


In [2]:
df = pd.read_csv(IN_PATH)
print("EDGE_Cal.csv shape", df.shape)
for  col in ["E_CBM", "E_VBM"]:
    assert col in df.columns, f"{col} column is missing from EDGE_Cal.csv"
df.head()



EDGE_Cal.csv shape (6794, 30)


,formula,elements,nelements,nsites,volume,density,crystal_system,space_group,formation_energy_per_atom,energy_above_hull,...,band_gap,band_gap_source,n_rare_elements,in_domain,gap_type,chi_mulliken,E_CBM,E_VBM,E_CBM_vac,E_VBM_vac
0,Ac2S3,Ac_S,2,40,1118.407852,6.535149,Tetragonal,I-42d,-2.493064,0.00000,...,1.964,ML,1,False,indirect,4.561109,-0.920891,1.043109,-3.579109,-5.543109
1,AcBrO,Ac_Br_O,3,6,140.139410,7.652290,Tetragonal,P4/nmm,-3.218186,0.00000,...,2.670,ML,1,False,direct,5.472917,-0.362083,2.307917,-4.137917,-6.807917
2,AcClO,Ac_Cl_O,3,6,129.213203,7.156871,Tetragonal,P4/nmm,-3.381758,0.00168,...,2.964,ML,1,False,indirect,5.636606,-0.345394,2.618606,-4.154606,-7.118606
3,AcCrO3,Ac_Cr_O,3,5,61.362845,8.848788,Cubic,Pm-3m,-3.138972,0.00000,...,1.541,ML,1,False,indirect,5.394781,0.124281,1.665281,-4.624281,-6.165281
4,AcH3,Ac_H,2,4,49.065716,7.784734,Cubic,Fm-3m,-0.671802,0.00000,...,2.353,ML,1,False,indirect,5.704421,0.027921,2.380921,-4.527921,-6.880921


In [3]:
cbm_ok = df["E_CBM"] <= CBM_MAX + TOL
vbm_ok = df["E_VBM"] >= VBM_MIN - TOL
keep = (cbm_ok & vbm_ok) if REQUIRE_VBM else cbm_ok

edge_df = df[keep].copy().reset_index(drop=True)

print("Screening funnel")
print(f"    EDGE_Cal.csv                       : {len(df):,}")
print(f"    E_CBM <= {CBM_MAX + TOL:.2f} V                 : {int(cbm_ok.sum()):,}")
print(f"    E_VBM >= {VBM_MIN - TOL:.2f} V                 : {int(vbm_ok.sum()):,}")
print(f"    kept (BandEdge.csv)                            : {len(edge_df)} ({len(edge_df) / len(df):.1%})")
print(f"    removed in total                    : {len(df) - len(edge_df):,}")
edge_df.head()

Screening funnel
    EDGE_Cal.csv                       : 6,794
    E_CBM <= 0.00 V                 : 4,577
    E_VBM >= 1.23 V                 : 5,169
    kept (BandEdge.csv)                            : 2952 (43.5%)
    removed in total                    : 3,842


,formula,elements,nelements,nsites,volume,density,crystal_system,space_group,formation_energy_per_atom,energy_above_hull,...,band_gap,band_gap_source,n_rare_elements,in_domain,gap_type,chi_mulliken,E_CBM,E_VBM,E_CBM_vac,E_VBM_vac
0,AcBrO,Ac_Br_O,3,6,140.139410,7.652290,Tetragonal,P4/nmm,-3.218186,0.00000,...,2.670,ML,1,False,direct,5.472917,-0.362083,2.307917,-4.137917,-6.807917
1,AcClO,Ac_Cl_O,3,6,129.213203,7.156871,Tetragonal,P4/nmm,-3.381758,0.00168,...,2.964,ML,1,False,indirect,5.636606,-0.345394,2.618606,-4.154606,-7.118606
2,AcI3,Ac_I,2,8,465.636416,4.334420,Hexagonal,P6_3/mmc,-1.801003,0.00000,...,2.527,ML,1,False,indirect,5.451448,-0.312052,2.214948,-4.187948,-6.714948
3,Ag2AsS2,Ag_As_S,3,40,881.720609,5.345361,Monoclinic,Pc,-0.146347,0.00303,...,1.730,ML,0,True,indirect,5.263092,-0.101908,1.628092,-4.398092,-6.128092
4,Ag2GeS3,Ag_Ge_S,3,12,267.880017,4.767775,Orthorhombic,Cmc2_1,-0.466994,0.00000,...,1.980,experiment,0,True,indirect,5.279274,-0.210726,1.769274,-4.289274,-6.269274


In [4]:
edge_df.to_csv(OUT_PATH, index=False)

summary = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": IN_PATH.name,
    "output_file": OUT_PATH.name,
    "edge_window_eV": [VBM_MIN, CBM_MAX],
    "row_in": int(len(df)),
    "rows_kept": int(len(edge_df)),
    

}
(DATA_DIR / f"{OUT_PATH.stem}_summary.json").write_text(json.dumps(summary, indent=2))
print("saved:", OUT_PATH)
print(json.dumps(summary, indent=2))

saved: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\BandEdge.csv
{
  "saved_at": "2026-11-03T20:54:45",
  "input_file": "EDGE_Cal.csv",
  "output_file": "BandEdge.csv",
  "edge_window_eV": [
    1.23,
    0.0
  ],
  "row_in": 6794,
  "rows_kept": 2952
}
